## Splitting and Embedding Text Using LangChain

In [36]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


with open('/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/churchill_speech.txt') as f:
    churchill_speech = f.read()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200,
        length_function = len
    )

In [22]:
chunks = text_splitter.split_text(churchill_speech)
print(len(chunks))

27


## Embedding Cost

In [23]:
def print_embedding_cost(docs):
    import tiktoken
    
    COST_PER_1K = 0.00002
    
    enc = tiktoken.encoding_for_model("text-embedding-3-small")
    
    total_tokens = sum(len(enc.encode(doc)) for doc in docs)
    
    cost = (total_tokens / 1000) * COST_PER_1K
    
    print(f"Total tokens: {total_tokens}")
    print(f"Estimated embedding cost in USD: ${cost:.6f}")

print_embedding_cost(chunks)

Total tokens: 5552
Estimated embedding cost in USD: $0.000111


In [49]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", 
    task_type="retrieval_document",
    output_dimensionality=1536
)

In [50]:
text = chunks[0]
vectors = embeddings.embed_documents([text])
vectors[:5]

[[-0.0066847582,
  0.014283699,
  0.02376309,
  -0.07331333,
  0.0049611675,
  0.03382253,
  0.007774566,
  -0.020077176,
  -0.018676357,
  -0.0003346588,
  -0.024206595,
  -0.014885024,
  0.008860484,
  -0.00049099,
  0.11222633,
  0.011405685,
  0.010005522,
  -0.02178445,
  0.009321887,
  0.00090838945,
  0.016369889,
  0.026454879,
  0.010117098,
  -0.0029657052,
  0.011788661,
  0.017059239,
  0.025312496,
  -0.008143399,
  0.027636502,
  -0.017082753,
  0.0037388562,
  0.004983235,
  0.0012835362,
  -0.010562962,
  -0.004787502,
  -0.0057154484,
  -0.0032166177,
  -0.010590599,
  -0.0103737125,
  0.016169762,
  0.010726594,
  -0.027949527,
  0.0017148599,
  -0.012005209,
  0.0037776588,
  -0.0066388436,
  0.0084184045,
  -0.024609149,
  -0.0037987572,
  0.010052209,
  -0.01847209,
  -0.0015924438,
  -0.013178114,
  -0.24425207,
  -0.0053405813,
  -0.0003351705,
  -0.0095181735,
  0.008650351,
  0.0039842315,
  0.011592335,
  -0.005248505,
  0.023887973,
  -0.01198943,
  -0.015598

In [45]:
import os 
import pinecone



pc = pinecone.Pinecone()

In [52]:
# deleting all indexes
indexes = pc.list_indexes().names()
for i in indexes:
    print('Deleting all indexes ... ', end='')
    pc.delete_index(i)
    print('Done')

Deleting all indexes ... Done


In [53]:
from pinecone import ServerlessSpec

index_name = 'churchill-speech'

if index_name not in pc.list_indexes().names():
    print(f'Creating index: {index_name}')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1',
        )

    )
    print(f'Index {index_name} created successfully.')
else:
    print(f'Index {index_name} already exists.')



Creating index: churchill-speech
Index churchill-speech created successfully.


In [54]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

docs = [Document(page_content=text) for text in chunks]

vectorstore = PineconeVectorStore.from_documents(
        docs,
        index_name=index_name,
        embedding=embeddings
    )

In [55]:
index = pc.Index(index_name)

index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 27}},
 'total_vector_count': 27,
 'vector_type': 'dense'}

In [57]:
query = 'Where should we fight?'
result = vectorstore.similarity_search(query)
print(result)

[Document(id='f48164ef-4cc4-464f-850d-9d526c487a5b', metadata={}, page_content="Majesty's Government-every man of them. That is the will of Parliament and the nation. The British\nEmpire and the French Republic, linked together in their cause and in their need, will defend to the\ndeath their native soil, aiding each other like good comrades to the utmost of their strength. Even\nthough large tracts of Europe and many old and famous States have fallen or may fall into the grip of\nthe Gestapo and all the odious apparatus of Nazi rule, we shall not flag or fail. We shall go on to the\nend, we shall fight in France, we shall fight on the seas and oceans, we shall fight with growing\nconfidence and growing strength in the air, we shall defend our Island, whatever the cost may be, we\nshall fight on the beaches, we shall fight on the landing grounds, we shall fight in the fields and in the\nstreets, we shall fight in the hills; we shall never surrender, and even if, which I do not for a mo

In [58]:
for r in result:
    print(r.page_content)
    print('-' * 50)

Majesty's Government-every man of them. That is the will of Parliament and the nation. The British
Empire and the French Republic, linked together in their cause and in their need, will defend to the
death their native soil, aiding each other like good comrades to the utmost of their strength. Even
though large tracts of Europe and many old and famous States have fallen or may fall into the grip of
the Gestapo and all the odious apparatus of Nazi rule, we shall not flag or fail. We shall go on to the
end, we shall fight in France, we shall fight on the seas and oceans, we shall fight with growing
confidence and growing strength in the air, we shall defend our Island, whatever the cost may be, we
shall fight on the beaches, we shall fight on the landing grounds, we shall fight in the fields and in the
streets, we shall fight in the hills; we shall never surrender, and even if, which I do not for a moment
--------------------------------------------------
Winston Churchill Speech - We Sh

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)


In [60]:
query = 'Answer only from the provided input. Where should we fight?'
answer = chain.invoke(query)
print(answer)

{'query': 'Answer only from the provided input. Where should we fight?', 'result': 'According to the text, we should fight:\n\n* In France\n* On the seas and oceans\n* In the air\n* On the beaches\n* On the landing grounds\n* In the fields\n* In the streets\n* In the hills \n\nWe shall defend our Island, whatever the cost may be.'}


In [61]:
query = 'Who was the king of Belgium at that time?'
answer = chain.invoke(query)
print(answer)

{'query': 'Who was the king of Belgium at that time?', 'result': 'The king of Belgium mentioned in the text is King Leopold. Specifically, it is likely referring to King Leopold III, who was the King of the Belgians from 1934 to 1951. During World War II, he played a significant role, and his decision to surrender the Belgian Army in 1940 is referenced in the provided text.'}


In [62]:
query = 'What about the French Armies??'
answer = chain.invoke(query)
print(answer)

{'query': 'What about the French Armies??', 'result': "According to the provided text, when the German penetration was realized, the French and British Armies in Belgium tried to hold their position and give their right hand to a newly created French Army that was to advance across the Somme in great strength. However, the German armored divisions cut off all communications between the Armies of the north and the main French Armies.\n\nIt is also mentioned that a new French Generalissimo, General Weygand, assumed command in place of General Gamelin, and an effort was made by the French and British Armies to hold their position. Additionally, French troops were able to hold the Graveline water lines after they were flooded, which helped to keep the port of Dunkirk open.\n\nBut overall, the text does not provide a detailed account of the French Armies' actions or fate, focusing more on the British perspective and the events surrounding the evacuation of Dunkirk."}
